# ChaoSSM — Phase 1d: Selective-Copy Variance Diagnostic

## Why this phase

Phase 1's selective-copy result was the one genuinely ambiguous outcome in the four-task
battery: `chaossm_free` finished slightly ahead of `linear` (+0.011) but not significantly
(p=0.125), and the free arm showed high seed variance (CV=0.34) — 5 seeds ranged
0.059-0.150, four clustered near chance (0.0625) with one substantially higher. v3.0 reported
this honestly as inconclusive rather than forcing it into a positive or negative reading.
This phase resolves it, or at least characterizes it properly, rather than leaving a guess in
the manuscript.

## Three questions, in order

1. **Is the variance bimodal** (a narrow solution basin some seeds find and others don't, the
   same shape as the parity investigation) **or continuous** (ordinary seed-to-seed noise)?
   Tested via Sarle's bimodality coefficient and, where available, a 1- vs 2-component
   Gaussian-mixture BIC comparison.
2. **Does more statistical power resolve the chaos-vs-linear comparison?** Phase 1 ran 5 seeds
   per arm; this phase runs 15, giving the paired Wilcoxon test real room to move off the n=5
   floor (min achievable p at n=15 is 2/2^15 ≈ 0.00006, vs. 0.0625 at n=5).
3. **Is the same learn-then-forget instability found in the parity investigation present here
   too**, or is it parity-specific? Tracked via best-vs-settled decay, exactly as in Phase
   1b/1c.

## Scope and stopping rule

Single task (selective copy), three arms (`linear`, `chaossm_free`, `chaossm_corridor`), 15
seeds each = 45 runs, same architecture and step budget as Phase 1. If this resolves the
question (clear bimodality + restored significance, or clean confirmation of a null result),
it closes here — consistent with the committed, non-open-ended framing adopted in v3.0. No
follow-up sub-phase is pre-planned; if the result is still ambiguous at n=15, that ambiguity is
reported as the finding rather than chased with a fourth iteration.

## Setup

In [1]:
import os, json, math, time, warnings
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from scipy.stats import wilcoxon, skew, kurtosis
from torch.func import jacrev

warnings.filterwarnings("ignore")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)

try:
    from sklearn.mixture import GaussianMixture
    HAVE_SKLEARN = True
except Exception:
    HAVE_SKLEARN = False
print("sklearn available for GMM check:", HAVE_SKLEARN)

def setup_results_dir(project="ChaoSSM", phase="phase1d"):
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        base = f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base = os.path.abspath(f"./{project}_results/{phase}")
        print(f"[info] Not in Colab ({type(e).__name__}); using local dir.")
    os.makedirs(base, exist_ok=True); print("results ->", base); return base

RESULTS = setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=300,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)
def savejson(o,n):
    p=os.path.join(RESULTS,n)
    with open(p,"w") as fh: json.dump(o,fh,indent=2,default=str)
    print("saved",p)

torch 2.11.0+cu128 | device: cuda
sklearn available for GMM check: True
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase1d


## Configuration

Task and architecture identical to Phase 1's selective-copy setting (`sel_len=24`,
`sel_content=6`, `steps=1200`). Only `seeds` changes (5 -> 15).

In [2]:
SMOKE_TEST = False   # full run: 3 arms x 15 seeds on selective_copy only

@dataclass
class Config:
    state_dim: int = 48
    d_model: int = 48
    vocab: int = 16
    steps: int = 1200
    batch: int = 64
    lr: float = 3e-4
    bptt_k: int = 64
    grad_clip: float = 1.0
    gamma_init: float = 1.0
    warmup_frac: float = 0.10
    dual_lr: float = 0.05
    alpha_max: float = 50.0
    corridor_target: float = 0.15
    corridor_half_width: float = 0.10
    lyap_window: int = 4
    lyap_subsample: int = 3
    lyap_every: int = 20
    sel_len: int = 24
    sel_content: int = 6
    arms: tuple = ("linear", "chaossm_free", "chaossm_corridor")
    seeds: int = 15
    eval_every: int = 20

CFG = Config()
if SMOKE_TEST:
    CFG.steps = 150; CFG.seeds = 2
    print("*** SMOKE TEST — pipeline check only ***")
print(json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

{
  "state_dim": "48",
  "d_model": "48",
  "vocab": "16",
  "steps": "1200",
  "batch": "64",
  "lr": "0.0003",
  "bptt_k": "64",
  "grad_clip": "1.0",
  "gamma_init": "1.0",
  "warmup_frac": "0.1",
  "dual_lr": "0.05",
  "alpha_max": "50.0",
  "corridor_target": "0.15",
  "corridor_half_width": "0.1",
  "lyap_window": "4",
  "lyap_subsample": "3",
  "lyap_every": "20",
  "sel_len": "24",
  "sel_content": "6",
  "arms": "('linear', 'chaossm_free', 'chaossm_corridor')",
  "seeds": "15",
  "eval_every": "20"
}


## 1. Model — unchanged from Phase 1

In [3]:
def triangle_fold(x):
    return 1.0 - torch.abs(torch.remainder(x, 2.0) - 1.0)

class LinearMap(nn.Module):
    def __init__(self, N):
        super().__init__()
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_lin = nn.Parameter(W * 0.5)
    def forward(self, h):
        return torch.tanh(h @ self.W_lin.T)

class ChaoticMap(nn.Module):
    def __init__(self, N, gamma_init=1.0):
        super().__init__()
        self.w_r = nn.Parameter(torch.zeros(N))
        W = torch.empty(N, N); nn.init.orthogonal_(W)
        self.W_mix = nn.Parameter(W * 0.1)
        inv = math.log(math.expm1(gamma_init))
        self.w_gamma = nn.Parameter(torch.tensor(float(inv)))
    def get_r(self): return 4.0 * torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def forward(self, h):
        r = self.get_r()
        u = h @ self.W_mix.T + r * h * (1 - h)
        return triangle_fold(self.get_gamma() * u)

class ChaoSSMCell(nn.Module):
    def __init__(self, cfg, phi_kind):
        super().__init__()
        self.N = cfg.state_dim
        self.phi_kind = phi_kind
        self.phi = LinearMap(cfg.state_dim) if phi_kind == "linear" \
                   else ChaoticMap(cfg.state_dim, cfg.gamma_init)
        self.A = nn.Parameter(torch.diag(torch.empty(cfg.state_dim).uniform_(0.9, 1.1)))
        self.B = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.W_g = nn.Parameter(torch.randn(cfg.state_dim, cfg.d_model) / math.sqrt(cfg.d_model))
        self.b_g = nn.Parameter(torch.zeros(cfg.state_dim))
    def step(self, h, x):
        g = torch.sigmoid(x @ self.W_g.T + self.b_g)
        raw = g * self.phi(h @ self.A.T) + (1 - g) * h + x @ self.B.T
        return triangle_fold(raw) if self.phi_kind != "linear" else torch.tanh(raw)
    def init_state(self, b, dev):
        return torch.zeros(b, self.N, device=dev) if self.phi_kind == "linear" \
               else torch.full((b, self.N), 0.5, device=dev)

class ChaoSSMLM(nn.Module):
    def __init__(self, cfg, phi_kind, vocab_size, readout_size):
        super().__init__()
        self.cfg = cfg
        self.emb = nn.Embedding(vocab_size, cfg.d_model)
        self.cell = ChaoSSMCell(cfg, phi_kind)
        self.readout = nn.Linear(cfg.state_dim, readout_size)
    def forward(self, tok, return_traj=False):
        B, T = tok.shape
        x = self.emb(tok); h = self.cell.init_state(B, tok.device)
        outs, traj = [], []
        for t in range(T):
            h = self.cell.step(h, x[:, t]); outs.append(self.readout(h))
            if return_traj: traj.append(h)
            if (t + 1) % self.cfg.bptt_k == 0: h = h.detach()
        lg = torch.stack(outs, 1)
        return (lg, torch.stack(traj, 1), x) if return_traj else lg

## 2. Estimators — unchanged

In [4]:
def _full_jac_fn(model, x):
    return lambda z: model.cell.step(z.unsqueeze(0), x.unsqueeze(0))[0]

def lambda_window(model, traj, xseq, W, n_sample):
    B, T, N = traj.shape
    if T <= W + 1:
        return torch.zeros((), device=traj.device)
    lams = []
    for b in torch.randperm(B)[:n_sample]:
        t0 = int(torch.randint(0, T - W - 1, (1,)))
        M = torch.eye(N, device=traj.device, dtype=traj.dtype)
        for k in range(W):
            M = jacrev(_full_jac_fn(model, xseq[b, t0+k]))(traj[b, t0+k]) @ M
        lams.append(torch.log(torch.linalg.matrix_norm(M, ord=2) + 1e-12) / W)
    return torch.stack(lams).mean()

## 3. Selective-copy task — identical to Phase 1

`L` positions, only `M` carry content tokens (rest are `blank`); reproduce the content tokens
in order after a delimiter. Every `torch.randint`/`torch.randperm` call that uses the seeded
generator explicitly passes `device=device` — this exact omission caused two separate crashes
in earlier phases, so it is checked here on every call site before this notebook is shipped.

In [5]:
def make_selective_copy_batch(cfg, batch, device, gen):
    L, M, V = cfg.sel_len, cfg.sel_content, cfg.vocab
    blank, sep, pad = V, V + 1, V + 2
    seq = torch.full((batch, L), blank, device=device)
    content = torch.randint(0, V, (batch, M), device=device, generator=gen)
    for b in range(batch):
        pos = torch.randperm(L, generator=gen, device=device)[:M].sort().values
        seq[b, pos] = content[b]
    inp = torch.cat([seq, torch.full((batch,1), sep, device=device),
                     torch.full((batch,M), pad, device=device)], 1)
    tgt = torch.cat([torch.full((batch, L+1), -100, device=device), content], 1)
    return inp, tgt

VOCAB_SIZE, READOUT_SIZE = CFG.vocab + 3, CFG.vocab + 3
CHANCE = 1.0 / CFG.vocab
print(f"chance accuracy = {CHANCE:.4f}")

chance accuracy = 0.0625


## 4. Trainer — unchanged mechanics, more seeds

In [6]:
def train_one(arm, seed, cfg):
    torch.manual_seed(seed); np.random.seed(seed)
    gen = torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    phi_kind = "linear" if arm == "linear" else "chaossm"
    model = ChaoSSMLM(cfg, phi_kind, VOCAB_SIZE, READOUT_SIZE).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, betas=(0.9, 0.95))

    use_lyap = (arm == "chaossm_corridor")
    lo = hi = None
    if use_lyap:
        lo = cfg.corridor_target - cfg.corridor_half_width
        hi = cfg.corridor_target + cfg.corridor_half_width
    a_hi = a_lo = 0.0
    warmup_steps = int(cfg.warmup_frac * cfg.steps)
    hist, diverged = [], False

    for step in range(cfg.steps):
        inp, tgt = make_selective_copy_batch(cfg, cfg.batch, DEVICE, gen)
        need_traj = (arm != "linear")
        out = model(inp, return_traj=need_traj)
        logits, traj, xseq = out if need_traj else (out, None, None)
        L_task = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), tgt.reshape(-1),
                                 ignore_index=-100)

        L_pen = torch.zeros((), device=DEVICE); lam_val = float("nan")
        if arm != "linear":
            in_warmup = step < warmup_steps
            lam = lambda_window(model, traj, xseq, cfg.lyap_window, cfg.lyap_subsample)
            lam_val = float(lam.detach())
            if use_lyap and not in_warmup:
                L_pen = a_hi * F.relu(lam - hi) + a_lo * F.relu(lo - lam)

        opt.zero_grad(set_to_none=True); (L_task + L_pen).backward()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip).item()
        opt.step()

        if use_lyap and step >= warmup_steps:
            a_hi = float(np.clip(a_hi + cfg.dual_lr * (lam_val - hi), 0.0, cfg.alpha_max))
            a_lo = float(np.clip(a_lo + cfg.dual_lr * (lo - lam_val), 0.0, cfg.alpha_max))

        if not (math.isfinite(L_task.item()) and math.isfinite(gn)):
            diverged = True; break

        if step % cfg.eval_every == 0:
            with torch.no_grad():
                mask = tgt != -100
                pred = logits.argmax(-1)
                acc = ((pred == tgt) & mask).float().sum().item() / mask.float().sum().clamp(min=1).item()
            hist.append(dict(step=step, arm=arm, seed=seed, loss=L_task.item(), acc=acc,
                             lam=lam_val, gnorm=gn))

    tail = hist[-max(1, len(hist)//5):]
    s = dict(arm=arm, seed=seed, diverged=diverged,
             final_acc=float(np.mean([h["acc"] for h in tail])) if tail else np.nan,
             best_acc=float(np.max([h["acc"] for h in hist])) if hist else np.nan,
             decay_from_peak=float(np.max([h["acc"] for h in hist]) -
                                   np.mean([h["acc"] for h in tail])) if hist else np.nan,
             lam_settled=float(np.nanmean([h["lam"] for h in tail])) if tail else np.nan)
    return s, pd.DataFrame(hist)

## 5. Run — 3 arms × 15 seeds on selective_copy

In [7]:
rows, hists = [], []
total = len(CFG.arms) * CFG.seeds
done = 0
for arm in CFG.arms:
    for sd in range(CFG.seeds):
        s, h = train_one(arm, sd, CFG)
        rows.append(s); hists.append(h); done += 1
        flag = " DIVERGED" if s["diverged"] else ""
        print(f"[{done}/{total}] {arm:18s} seed={sd:2d}  final={s['final_acc']:.4f} "
              f"(chance {CHANCE:.4f})  best={s['best_acc']:.4f}  decay={s['decay_from_peak']:.4f}  "
              f"lam={s['lam_settled']:+.3f}{flag}")

SUM = pd.DataFrame(rows); HIST = pd.concat(hists, ignore_index=True)
savetable(SUM, "phase1d_summary.csv"); savetable(HIST, "phase1d_history.csv")

[1/45] linear             seed= 0  final=0.0638 (chance 0.0625)  best=0.1016  decay=0.0378  lam=+nan
[2/45] linear             seed= 1  final=0.0592 (chance 0.0625)  best=0.0859  decay=0.0267  lam=+nan
[3/45] linear             seed= 2  final=0.0668 (chance 0.0625)  best=0.0938  decay=0.0269  lam=+nan
[4/45] linear             seed= 3  final=0.0684 (chance 0.0625)  best=0.0938  decay=0.0254  lam=+nan
[5/45] linear             seed= 4  final=0.0816 (chance 0.0625)  best=0.0938  decay=0.0122  lam=+nan
[6/45] linear             seed= 5  final=0.0597 (chance 0.0625)  best=0.0938  decay=0.0341  lam=+nan
[7/45] linear             seed= 6  final=0.1048 (chance 0.0625)  best=0.1458  decay=0.0410  lam=+nan
[8/45] linear             seed= 7  final=0.0681 (chance 0.0625)  best=0.0938  decay=0.0256  lam=+nan
[9/45] linear             seed= 8  final=0.0933 (chance 0.0625)  best=0.1146  decay=0.0213  lam=+nan
[10/45] linear             seed= 9  final=0.1003 (chance 0.0625)  best=0.1276  decay=0.0273

## 6. Question 1 — is the variance bimodal?

Sarle's bimodality coefficient: BC = (skew² + 1) / kurtosis (excess kurtosis + 3 convention).
BC > 0.555 is the standard rule-of-thumb threshold suggesting bimodality for a finite sample
(0.555 is the value for a uniform distribution; unimodal distributions like the normal score
below it). Where scikit-learn is available, a 1- vs. 2-component Gaussian-mixture BIC
comparison is run as a second, independent check — a lower BIC for 2 components is corroborating
evidence, not required for the conclusion.

In [8]:
def bimodality_coefficient(x):
    x = np.asarray(x, dtype=np.float64)
    n = len(x)
    g = skew(x, bias=False)
    k = kurtosis(x, fisher=False, bias=False)  # Pearson kurtosis (normal = 3)
    return (g**2 + 1) / (k + 3*(n-1)**2 / ((n-2)*(n-3))) if n > 3 else np.nan

bimodal_rows = []
for arm in CFG.arms:
    vals = SUM[SUM.arm == arm].final_acc.dropna().values
    bc = bimodality_coefficient(vals)
    row = dict(arm=arm, n=len(vals), bimodality_coef=float(bc) if np.isfinite(bc) else None,
              likely_bimodal=bool(np.isfinite(bc) and bc > 0.555),
              mean=float(vals.mean()), std=float(vals.std()), cv=float(vals.std()/vals.mean()) if vals.mean() else None)
    if HAVE_SKLEARN and len(vals) >= 6:
        X = vals.reshape(-1, 1)
        bic1 = GaussianMixture(1, random_state=0).fit(X).bic(X)
        bic2 = GaussianMixture(2, random_state=0, n_init=5).fit(X).bic(X)
        row["gmm_bic_1comp"] = float(bic1); row["gmm_bic_2comp"] = float(bic2)
        row["gmm_favors_2comp"] = bool(bic2 < bic1)
    bimodal_rows.append(row)

BIMODAL = pd.DataFrame(bimodal_rows)
print(BIMODAL.round(4).to_string(index=False))
savetable(BIMODAL, "phase1d_bimodality.csv")

             arm  n  bimodality_coef  likely_bimodal   mean    std     cv  gmm_bic_1comp  gmm_bic_2comp  gmm_favors_2comp
          linear 15           0.3356           False 0.0733 0.0148 0.2025       -78.3335       -86.1957              True
    chaossm_free 15           0.2711           False 0.1062 0.0434 0.4088       -46.1155       -47.5811              True
chaossm_corridor 15           0.2190           False 0.1382 0.0450 0.3255       -45.0729       -46.2782              True
saved /content/drive/My Drive/ChaoSSM/results/phase1d/phase1d_bimodality.csv


In [9]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4.2))
for i, arm in enumerate(CFG.arms):
    vals = SUM[SUM.arm == arm].final_acc.dropna().values
    ax[i].hist(vals, bins=max(5, len(vals)//2), color=["gray","tab:blue","tab:green"][i], alpha=.8)
    ax[i].axvline(CHANCE, color="r", ls="--", label="chance")
    ax[i].axvline(vals.mean(), color="k", ls="-", label="mean")
    bc = BIMODAL[BIMODAL.arm==arm].bimodality_coef.values[0]
    ax[i].set(title=f"{arm}\nBC={bc:.3f}" if bc is not None else arm,
             xlabel="final accuracy", ylabel="count" if i==0 else "")
    ax[i].legend(fontsize=7); ax[i].grid(alpha=.3)
fig.tight_layout(); savefig(fig, "phase1d_distributions.png"); plt.close(fig)

saved /content/drive/My Drive/ChaoSSM/results/phase1d/phase1d_distributions.png


## 7. Question 2 — does more power resolve chaos vs. linear?

Same paired-Wilcoxon protocol as Phase 1, now at n=15. The minimum achievable two-sided
p-value at n=15 is 2/2^15 ≈ 0.00006 — dramatically more room than n=5's 0.0625 floor for a
real effect to actually register as significant, if one exists.

In [10]:
def paired(a, b, col="final_acc"):
    x = SUM[SUM.arm==a].sort_values("seed")[col].values
    y = SUM[SUM.arm==b].sort_values("seed")[col].values
    m = np.isfinite(x) & np.isfinite(y); x, y = x[m], y[m]
    if len(x) < 3 or np.allclose(x, y):
        return dict(pair=f"{a} vs {b}", n=len(x), p=np.nan,
                    median_diff=float(np.median(x-y)) if len(x) else np.nan)
    st, p = wilcoxon(x, y)
    return dict(pair=f"{a} vs {b}", n=len(x), p=float(p), median_diff=float(np.median(x-y)))

ST = pd.DataFrame([paired("chaossm_free","linear"), paired("chaossm_corridor","linear"),
                   paired("chaossm_corridor","chaossm_free")])
print(ST.to_string(index=False))
savetable(ST, "phase1d_stats.csv")
print(f"\nmin achievable two-sided p at n={CFG.seeds}: {2/(2**CFG.seeds):.6f}")
print(SUM.groupby("arm")[["final_acc","best_acc","decay_from_peak"]].agg(["mean","std"]).round(4).to_string())

                            pair  n        p  median_diff
          chaossm_free vs linear 15 0.012436     0.024306
      chaossm_corridor vs linear 15 0.002158     0.078776
chaossm_corridor vs chaossm_free 15 0.018066     0.016927
saved /content/drive/My Drive/ChaoSSM/results/phase1d/phase1d_stats.csv

min achievable two-sided p at n=15: 0.000061
                 final_acc         best_acc         decay_from_peak        
                      mean     std     mean     std            mean     std
arm                                                                        
chaossm_corridor    0.1382  0.0465   0.1750  0.0396          0.0368  0.0144
chaossm_free        0.1062  0.0450   0.1420  0.0438          0.0358  0.0155
linear              0.0733  0.0154   0.1043  0.0164          0.0311  0.0088


## 8. Question 3 — is the parity-style instability present here too?

Compare `decay_from_peak` here against the 0.162 ± 0.012 figure established across 12
conditions in the parity investigation. A similar magnitude would suggest the same
architecture-independent optimization phenomenon; a much smaller value would suggest
selective-copy's variance has a different origin.

In [11]:
decay_by_arm = SUM.groupby("arm").decay_from_peak.agg(["mean","std"])
print("decay from peak (selective_copy) vs. parity's cross-condition figure (0.162 +/- 0.012):")
print(decay_by_arm.round(4).to_string())

fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))
SUM.boxplot(column="decay_from_peak", by="arm", ax=ax[0])
ax[0].axhline(0.162, color="purple", ls="--", label="parity reference (0.162)")
ax[0].set(title="Decay from peak", xlabel="arm", ylabel="decay"); ax[0].legend()

for arm in ["chaossm_free", "chaossm_corridor"]:
    d = SUM[SUM.arm==arm]
    ax[1].scatter(d.lam_settled, d.final_acc, label=arm, alpha=.7)
ax[1].axvspan(0.0, 0.4, color="green", alpha=.1, label="0d plateau [0,0.4]")
ax[1].set(xlabel=r"settled $\lambda$", ylabel="final accuracy",
          title="Does lambda predict outcome within the chaotic arms?")
ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
fig.suptitle(""); fig.tight_layout(); savefig(fig, "phase1d_decay_and_lambda.png"); plt.close(fig)

corr_free = SUM[SUM.arm=="chaossm_free"][["lam_settled","final_acc"]].corr().iloc[0,1]
print(f"\ncorr(lambda_settled, final_acc) within chaossm_free: {corr_free:.3f}")

decay from peak (selective_copy) vs. parity's cross-condition figure (0.162 +/- 0.012):
                    mean     std
arm                             
chaossm_corridor  0.0368  0.0144
chaossm_free      0.0358  0.0155
linear            0.0311  0.0088
saved /content/drive/My Drive/ChaoSSM/results/phase1d/phase1d_decay_and_lambda.png

corr(lambda_settled, final_acc) within chaossm_free: -0.595


## 9. Verdict

| Observation | Reading |
|---|---|
| BC > 0.555 and/or GMM favors 2 components, AND n=15 restores significance | Confirmed bimodal narrow-basin effect, same shape as the parity investigation but here some seeds *do* find the good basin. Report the effect as real but seed-sensitive; recommend more seeds as standard practice for this task in any future study using it. |
| BC low, unimodal, n=15 still not significant | The Phase 1 result was ordinary sampling noise around a true near-null effect. Report selective_copy as a confirmed null result — chaos does not reliably help here — rather than inconclusive. |
| decay_from_peak ≈ 0.16, matching parity | The same architecture-independent instability affects this task too; note it as a general property of the training recipe at this scale, not selective-copy-specific. |
| decay_from_peak much smaller than parity | Selective-copy's variance has a different origin than parity's collapse — consistent with a genuine seed-sensitivity/basin story rather than a generic optimization artifact. |
| corr(lambda, accuracy) within chaossm_free is strong | Performance is directly explained by where each seed's λ happened to settle — ties this result back to the S5.2 plateau finding directly and gives a mechanistic account, not just a statistical one. |

Whatever the combination, this phase closes the selective-copy question per the stopping rule
stated at the outset — the result (bimodal-and-real, or null-and-confirmed) goes into the
manuscript as stated, without a further follow-up phase.

In [12]:
verdict = dict(
    smoke_test=SMOKE_TEST,
    bimodality=BIMODAL.to_dict("records"),
    stats=ST.to_dict("records"),
    decay_by_arm=decay_by_arm.round(4).to_dict("index"),
    parity_reference_decay=0.162,
    corr_lambda_accuracy_chaossm_free=float(corr_free),
    config=dict(steps=CFG.steps, seeds=CFG.seeds, sel_len=CFG.sel_len, sel_content=CFG.sel_content),
)
savejson(verdict, "phase1d_verdict.json")
print(json.dumps(verdict, indent=2, default=str))

saved /content/drive/My Drive/ChaoSSM/results/phase1d/phase1d_verdict.json
{
  "smoke_test": false,
  "bimodality": [
    {
      "arm": "linear",
      "n": 15,
      "bimodality_coef": 0.335557190798935,
      "likely_bimodal": false,
      "mean": 0.07326388888888886,
      "std": 0.014837530170242452,
      "cv": 0.2025217388165795,
      "gmm_bic_1comp": -78.33345435985419,
      "gmm_bic_2comp": -86.19568888192289,
      "gmm_favors_2comp": true
    },
    {
      "arm": "chaossm_free",
      "n": 15,
      "bimodality_coef": 0.271145662316218,
      "likely_bimodal": false,
      "mean": 0.1062355324074074,
      "std": 0.043427717564579305,
      "cv": 0.4087871221658344,
      "gmm_bic_1comp": -46.11546313954305,
      "gmm_bic_2comp": -47.58107412658092,
      "gmm_favors_2comp": true
    },
    {
      "arm": "chaossm_corridor",
      "n": 15,
      "bimodality_coef": 0.21899660064962065,
      "likely_bimodal": false,
      "mean": 0.13815104166666667,
      "std": 0.044963